# Flood Risk Prediction Workflow 
This cleaned version keeps only the important steps from the original notebook:

1. Load and rename the four Auckland area datasets  
2. Clean missing values and obvious outliers  
3. Create rainfall, lag, seasonal, and event-level features  
4. Detect and rank flood events using the 99th percentile river-level threshold  
5. Train baseline models and compare performance  
6. Run SHAP explainability for the best model  
7. Build a unified hybrid model  
8. Validate the model on selected real flood/storm events  

Repeated plotting, duplicate model-training blocks, and multiple versions of the same styled table code have been removed.

## 1. Import libraries and define settings

This section imports all required libraries once and defines the file paths, area names, and manually verified column names.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False

RANDOM_STATE = 42

files = {
    "central": "central.csv",
    "albany": "Albany.csv",
    "henderson": "Henderson.csv",
    "manukau": "manukau_full.csv"
}

area_labels = {
    "central": "R1 (Central Auckland)",
    "albany": "R2 (Albany)",
    "henderson": "R3 (Henderson)",
    "manukau": "R4 (Manukau)"
}

final_column_names = {
    "central": [
        "time",
        "alexandra_park_rainfall_mm",
        "mt_roskill_substation_rainfall_mm",
        "mt_albert_grammar_rainfall_mm",
        "oakley_creek_richardson_road_discharge_m3s",
        "oakley_creek_richardson_road_level_m",
        "penrose_wind_speed_ms"
    ],
    "albany": [
        "time",
        "albany_heights_road_rainfall_mm",
        "awanohi_okura_rainfall_mm",
        "oteha_river_days_bridge_discharge_m3s",
        "oteha_river_days_bridge_level_m",
        "takapuna_wind_speed_ms"
    ],
    "henderson": [
        "start_time",
        "end_time",
        "keeling_road_rainfall_mm",
        "te_pai_park_henderson_rainfall_mm",
        "opanuku_vintage_reserve_discharge_m3s",
        "opanuku_vintage_reserve_level_m",
        "henderson_wind_speed_ms"
    ],
    "manukau": [
        "start_time",
        "end_time",
        "puhinui_botanics_rainfall_mm",
        "manukau_sports_bowl_rainfall_mm",
        "puhinui_drop_structure_level_m",
        "puhinui_drop_structure_discharge_m3s",
        "papatoetoe_wind_speed_ms"
    ]
}

## 2. Load and clean datasets

The functions below load each CSV file, apply meaningful column names, convert time and numeric columns, remove invalid values, and interpolate missing river/discharge/wind values while treating missing rainfall as zero.

In [ ]:
def get_time_col(df):
    if "start_time" in df.columns:
        return "start_time"
    if "time" in df.columns:
        return "time"
    raise ValueError("No time column found.")


def first_col(df, keywords, required=True):
    cols = [c for c in df.columns if all(k.lower() in c.lower() for k in keywords)]
    if cols:
        return cols[0]
    if required:
        raise ValueError(f"No column found for keywords: {keywords}")
    return None


def load_and_rename(area, path):
    df = pd.read_csv(path, header=4)
    df.columns = final_column_names[area]

    for col in ["time", "start_time", "end_time"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    for col in df.columns:
        if col not in ["time", "start_time", "end_time"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def clean_dataset(df, river_upper_limit=6):
    df = df.copy()
    time_col = get_time_col(df)

    rainfall_cols = [c for c in df.columns if "rainfall" in c.lower()]
    river_col = first_col(df, ["level"])
    discharge_cols = [c for c in df.columns if "discharge" in c.lower()]
    wind_cols = [c for c in df.columns if "wind" in c.lower() and "speed" in c.lower()]

    df = df.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)

# Remove impossible values before interpolation.
    for col in rainfall_cols:
        df.loc[df[col] < 0, col] = np.nan
        df[col] = df[col].fillna(0)

    df.loc[(df[river_col] < 0) | (df[river_col] > river_upper_limit), river_col] = np.nan

    for col in discharge_cols:
        df.loc[df[col] < 0, col] = np.nan

    for col in wind_cols:
        df.loc[df[col] < 0, col] = np.nan

# Time-based interpolation for non-rainfall numeric columns.
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    interp_cols = [c for c in numeric_cols if c not in rainfall_cols]

    df = df.set_index(time_col)
    df[interp_cols] = df[interp_cols].interpolate(method="time", limit_direction="both")
    df[interp_cols] = df[interp_cols].ffill().bfill()
    df = df.reset_index()

    return df


cleaned_datasets = {}

for area, path in files.items():
    raw_df = load_and_rename(area, path)
    cleaned_datasets[area] = clean_dataset(raw_df)
    print(f"{area}: {cleaned_datasets[area].shape[0]} rows, {cleaned_datasets[area].shape[1]} columns")

## 3. Feature engineering

This section creates rainfall accumulation windows, river-level lag variables, and time-based features used by the flood prediction models.

In [ ]:
def choose_rainfall_column(df, area):
    rainfall_cols = [c for c in df.columns if "rainfall" in c.lower()]
    if not rainfall_cols:
        return None

# Prefer the rainfall station matching the area name; otherwise use the column with the highest total rainfall.
    preferred = [c for c in rainfall_cols if area.lower() in c.lower()]
    if preferred:
        return preferred[0]

    totals = {c: pd.to_numeric(df[c], errors="coerce").fillna(0).sum() for c in rainfall_cols}
    return max(totals, key=totals.get)


def add_features(df, area):
    df = df.copy()
    time_col = get_time_col(df)
    river_col = first_col(df, ["level"])
    rain_col = choose_rainfall_column(df, area)

    if rain_col is not None:
        rain = pd.to_numeric(df[rain_col], errors="coerce").fillna(0)
        for window in [1, 3, 6, 12, 24, 48, 72]:
            df[f"rain_{window}h"] = rain.rolling(window, min_periods=1).sum()
    else:
        for window in [1, 3, 6, 12, 24, 48, 72]:
            df[f"rain_{window}h"] = np.nan

    for lag in [1, 3, 6]:
        df[f"river_lag_{lag}"] = df[river_col].shift(lag)

    df["month"] = df[time_col].dt.month
    df["hour"] = df[time_col].dt.hour
    df["season"] = (df["month"] % 12 // 3) + 1

    return df


for area in cleaned_datasets:
    cleaned_datasets[area] = add_features(cleaned_datasets[area], area)

## 4. Detect and rank flood events

Flood events are identified using the 99th percentile river-level threshold for each area. Consecutive threshold exceedances are grouped into flood events, then ranked by peak river water level.

In [ ]:
def detect_ranked_flood_events(df, area, quantile=0.99):
    df = df.copy()
    time_col = get_time_col(df)
    river_col = first_col(df, ["level"])
    discharge_col = first_col(df, ["discharge"], required=False)
    wind_col = first_col(df, ["wind", "speed"], required=False)
    rain_col = choose_rainfall_column(df, area)

    threshold = df[river_col].quantile(quantile)
    df["flood_flag"] = df[river_col] >= threshold
    df["event_group"] = (df["flood_flag"] != df["flood_flag"].shift()).cumsum()

    events = []

    for _, event_df in df.groupby("event_group"):
        if not bool(event_df["flood_flag"].iloc[0]):
            continue

        start = event_df[time_col].iloc[0]
        end = event_df[time_col].iloc[-1]
        duration = (end - start).total_seconds() / 3600

        row = {
            "area": area,
            "Region": area_labels.get(area, area),
            "start": start,
            "end": end,
            "duration_hours": duration,
            "threshold": threshold,
            "peak_water_level": event_df[river_col].max(),
            "mean_water_level": event_df[river_col].mean(),
            "river_level_at_start": event_df[river_col].iloc[0],
            "total_event_rainfall": event_df[rain_col].sum() if rain_col else np.nan,
            "peak_rainfall_intensity": event_df[rain_col].max() if rain_col else np.nan,
            "antecedent_rain_24h": df.loc[(df[time_col] >= start - pd.Timedelta(hours=24)) & (df[time_col] < start), rain_col].sum() if rain_col else np.nan,
            "antecedent_rain_48h": df.loc[(df[time_col] >= start - pd.Timedelta(hours=48)) & (df[time_col] < start), rain_col].sum() if rain_col else np.nan,
            "antecedent_rain_72h": df.loc[(df[time_col] >= start - pd.Timedelta(hours=72)) & (df[time_col] < start), rain_col].sum() if rain_col else np.nan,
            "peak_discharge": event_df[discharge_col].max() if discharge_col else np.nan,
            "peak_wind_speed": event_df[wind_col].max() if wind_col else np.nan,
            "month": start.month,
            "season": (start.month % 12 // 3) + 1,
        }
        events.append(row)

    events_df = pd.DataFrame(events)
    if events_df.empty:
        return events_df

    events_df = events_df.sort_values("peak_water_level", ascending=False).reset_index(drop=True)
    events_df["rank"] = np.arange(1, len(events_df) + 1)
    return events_df


all_flood_results = {}
all_events = []

for area, df in cleaned_datasets.items():
    events_df = detect_ranked_flood_events(df, area)
    all_flood_results[area] = events_df
    all_events.append(events_df)
    print(f"{area}: {len(events_df)} flood events detected")

all_events_df = pd.concat(all_events, ignore_index=True)
top5_events_df = all_events_df[all_events_df["rank"] <= 5].copy()

top5_events_df[[
    "Region", "rank", "start", "end", "duration_hours", "peak_water_level",
    "total_event_rainfall", "antecedent_rain_24h", "peak_discharge", "peak_wind_speed"
]].round(3)

## 5. Create a clean top-5 flood event table

This table is suitable for report writing. It keeps the main event features and removes repeated styled-table versions from the original notebook.

In [ ]:
event_table_df = top5_events_df.rename(columns={
    "rank": "Flood Event Rank",
    "start": "Start",
    "end": "End",
    "duration_hours": "Duration (hrs)",
    "peak_water_level": "Peak Water Level",
    "river_level_at_start": "River Level at Start",
    "total_event_rainfall": "Total Rainfall",
    "peak_rainfall_intensity": "Peak Rainfall Intensity",
    "antecedent_rain_24h": "Antecedent Rainfall 24h",
    "antecedent_rain_48h": "Antecedent Rainfall 48h",
    "antecedent_rain_72h": "Antecedent Rainfall 72h",
    "peak_discharge": "Peak Discharge",
    "peak_wind_speed": "Peak Wind Speed",
})

columns_to_keep = [
    "Region", "Flood Event Rank", "Start", "End", "Duration (hrs)",
    "Peak Water Level", "River Level at Start", "Total Rainfall", "Peak Rainfall Intensity",
    "Antecedent Rainfall 24h", "Antecedent Rainfall 48h", "Antecedent Rainfall 72h",
    "Peak Discharge", "Peak Wind Speed", "season"
]

event_table_df = event_table_df[columns_to_keep].copy()

event_table_df["Start"] = pd.to_datetime(event_table_df["Start"]).dt.strftime("%Y-%m-%d %H:%M")
event_table_df["End"] = pd.to_datetime(event_table_df["End"]).dt.strftime("%Y-%m-%d %H:%M")

event_table_df.round(3)

## 6. Baseline model development

Each area is modelled separately. The target is defined as the top-5 ranked flood events (`rank <= 5`). This keeps the same event-level logic but removes repeated training blocks.

In [ ]:
model_features = [
    "duration_hours",
    "peak_water_level",
    "mean_water_level",
    "river_level_at_start",
    "total_event_rainfall",
    "peak_rainfall_intensity",
    "antecedent_rain_24h",
    "antecedent_rain_48h",
    "antecedent_rain_72h",
    "peak_discharge",
    "peak_wind_speed",
    "month",
    "season"
]


def get_baseline_models():
    models = {
        "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
        "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=RANDOM_STATE),
        "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    }
    if XGBOOST_AVAILABLE:
        models["XGBoost"] = XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            eval_metric="logloss",
            random_state=RANDOM_STATE
        )
    return models


def evaluate_classifier(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else y_pred

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob) if len(np.unique(y_test)) > 1 else np.nan,
    }


baseline_results = []
best_models_by_area = {}

for area, events_df in all_flood_results.items():
    if events_df.empty or events_df["rank"].nunique() < 2:
        continue

    df_model = events_df.copy()
    df_model["target"] = (df_model["rank"] <= 5).astype(int)

    X = df_model[model_features].apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median(numeric_only=True))
    y = df_model["target"]

    stratify_y = y if y.nunique() == 2 and y.value_counts().min() >= 2 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=stratify_y
    )

    area_best_model = None
    area_best_score = -np.inf

    for model_name, model in get_baseline_models().items():
        model.fit(X_train, y_train)
        metrics = evaluate_classifier(model, X_test, y_test)

        baseline_results.append({
            "Area": area_labels.get(area, area),
            "Model": model_name,
            **metrics
        })

        if metrics["ROC-AUC"] > area_best_score:
            area_best_score = metrics["ROC-AUC"]
            area_best_model = model

    best_models_by_area[area] = area_best_model

baseline_results_df = pd.DataFrame(baseline_results)
baseline_results_df.round(3)

## 7. SHAP explainability

Run SHAP for the best model in each area. If SHAP is not installed, this cell will skip safely.

In [ ]:
# BEST MODEL NAMES BY AREA
best_model_names_by_area = {
    "central": "Gradient Boosting",
    "albany": "Random Forest",
    "henderson": "Random Forest",
    "manukau": "Logistic Regression"
}

In [ ]:
areas = ["central", "albany", "henderson", "manukau"]

shap_results_by_area = {}

for area in areas:


# Check required model objects
    if area not in best_models_by_area:
        print(f"Skipping {area}: no trained model found.")
        continue

    print("\n" + "=" * 80)
    print(f"SHAP ANALYSIS - {area.upper()}")
    print("=" * 80)

    df_model = all_flood_results[area].copy()
    model = best_models_by_area[area]
    model_name = best_model_names_by_area[area]
    features = model_features


# Prepare feature matrix
    df_model["target"] = (df_model["rank"] <= 5).astype(int)

    X = df_model[features].apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median(numeric_only=True))
    
# Create SHAP explainer
    try:
        if model_name in ["Decision Tree", "Random Forest", "Gradient Boosting", "XGBoost"]:
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(X)

            if isinstance(shap_values, list):
                shap_values_plot = np.array(shap_values[1])
            else:
                shap_values_plot = np.array(shap_values)

        elif model_name == "Logistic Regression":
            explainer = shap.Explainer(model, X)
            shap_values = explainer(X)
            shap_values_plot = np.array(shap_values.values)

        else:
            print(f"Skipping {area}: SHAP not configured for {model_name}.")
            continue

    except Exception as e:
        print(f"SHAP failed for {area}: {e}")
        continue

# Fix SHAP output shape for plotting
    
    if shap_values_plot.ndim == 3:
        shap_values_plot = shap_values_plot[:, :, 1]

    elif shap_values_plot.ndim == 1:
        shap_values_plot = shap_values_plot.reshape(-1, 1)

# Check SHAP values match feature count
    
    if shap_values_plot.shape[1] != len(features):
        print(
            f"Skipping {area}: SHAP-feature mismatch "
            f"({shap_values_plot.shape[1]} SHAP columns vs {len(features)} features)."
        )
        continue

# Calculate mean absolute SHAP importance
    
    shap_importance = pd.DataFrame({
        "Feature": features,
        "Mean |SHAP|": np.abs(shap_values_plot).mean(axis=0)
    }).sort_values("Mean |SHAP|", ascending=False).reset_index(drop=True)

    shap_results_by_area[area] = shap_importance

    print("\nTop SHAP drivers:")
    print(shap_importance)

# Plot 1: Mean SHAP importance bar chart
    
    plt.figure(figsize=(8, 5))
    plt.barh(
        shap_importance["Feature"][::-1],
        shap_importance["Mean |SHAP|"][::-1]
    )
    plt.title(f"{area.upper()} - SHAP Feature Importance ({model_name})")
    plt.xlabel("Mean Absolute SHAP Value")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

# Plot 2: SHAP summary plot
    
    shap.summary_plot(shap_values_plot, X, show=False)
    plt.title(f"{area.upper()} - SHAP Summary Plot ({model_name})")
    plt.tight_layout()
    plt.show()
    
# Plot 3: SHAP summary bar plot
    
    shap.summary_plot(shap_values_plot, X, plot_type="bar", show=False)
    plt.title(f"{area.upper()} - SHAP Summary Bar Plot ({model_name})")
    plt.tight_layout()
    plt.show()

## 8. Unified hybrid model

The hybrid model combines Random Forest, Decision Tree, and Logistic Regression using stacking. Area is included as an encoded feature to support multi-region prediction.

In [ ]:
hybrid_df = []

for area, events_df in all_flood_results.items():
    df_area = events_df.copy()
    df_area["area"] = area
    df_area["target"] = (df_area["rank"] <= 5).astype(int)
    hybrid_df.append(df_area)

hybrid_df = pd.concat(hybrid_df, ignore_index=True)

le_area = LabelEncoder()
hybrid_df["area_encoded"] = le_area.fit_transform(hybrid_df["area"])

hybrid_features = model_features + ["area_encoded"]

X = hybrid_df[hybrid_features].apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median(numeric_only=True))
y = hybrid_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)

base_models = [
    ("rf", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=RANDOM_STATE)),
    ("dt", DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
    ("lr", LogisticRegression(max_iter=2000, class_weight="balanced")),
]

hybrid_model = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(max_iter=2000, class_weight="balanced"),
    passthrough=True,
    cv=3
)

hybrid_model.fit(X_train, y_train)
hybrid_metrics = evaluate_classifier(hybrid_model, X_test, y_test)

pd.DataFrame([hybrid_metrics]).round(3)

## 9. Area-wise hybrid model evaluation

This checks how the unified hybrid model performs separately for each Auckland area.

In [ ]:
area_results = []

for area, events_df in all_flood_results.items():
    df_area = events_df.copy()
    df_area["area"] = area
    df_area["target"] = (df_area["rank"] <= 5).astype(int)
    df_area["area_encoded"] = le_area.transform(df_area["area"])

    X_area = df_area[hybrid_features].apply(pd.to_numeric, errors="coerce")
    X_area = X_area.fillna(X_area.median(numeric_only=True))
    y_area = df_area["target"]

    metrics = evaluate_classifier(hybrid_model, X_area, y_area)
    area_results.append({"Area": area_labels.get(area, area), **metrics})

area_results_df = pd.DataFrame(area_results)
area_results_df.round(3)

## 10. Define reusable event-based validation functions

This section prepares a reusable validation function. The same function is used for both the January 2023 Auckland Anniversary Flood and the May 2023 Auckland storm event.


In [ ]:
def build_timestamp_dataset(cleaned_datasets, validation_start, validation_end):
    rows = []

    for area, df in cleaned_datasets.items():
        df = df.copy()
        time_col = get_time_col(df)
        river_col = first_col(df, ["level"])
        discharge_col = first_col(df, ["discharge"], required=False)
        wind_col = first_col(df, ["wind", "speed"], required=False)
        rain_col = choose_rainfall_column(df, area)

        threshold = df[river_col].quantile(0.99)
        df["actual_flood"] = (df[river_col] >= threshold).astype(int)

        out = pd.DataFrame({
            "time": df[time_col],
            "area": area,
            "river_level": df[river_col],
            "actual_flood": df["actual_flood"],
            "flood_level_threshold": threshold,
            "month": df[time_col].dt.month,
            "hour": df[time_col].dt.hour,
            "discharge": df[discharge_col] if discharge_col else np.nan,
            "wind_speed": df[wind_col] if wind_col else np.nan,
        })

        if rain_col:
            rain = pd.to_numeric(df[rain_col], errors="coerce").fillna(0)
            for window in [3, 6, 12, 24]:
                out[f"rain_{window}h"] = rain.rolling(window, min_periods=1).sum()
        else:
            for window in [3, 6, 12, 24]:
                out[f"rain_{window}h"] = np.nan

        for lag in [1, 3, 6]:
            out[f"river_lag_{lag}"] = df[river_col].shift(lag)

        rows.append(out)

    df_all = pd.concat(rows, ignore_index=True)
    df_all["area_encoded"] = LabelEncoder().fit_transform(df_all["area"])

    return df_all


def validate_storm_event(cleaned_datasets, test_start, test_end, decision_threshold=0.50):
    df_all = build_timestamp_dataset(cleaned_datasets, test_start, test_end)

    features = [
        "rain_3h", "rain_6h", "rain_12h", "rain_24h",
        "river_lag_1", "river_lag_3", "river_lag_6",
        "month", "hour", "discharge", "wind_speed", "area_encoded"
    ]

    df_model = df_all.dropna(subset=["time", "actual_flood"]).copy()

    train_mask = df_model["time"] < test_start
    test_mask = (df_model["time"] >= test_start) & (df_model["time"] <= test_end)

    X = df_model[features].apply(pd.to_numeric, errors="coerce")
    y = df_model["actual_flood"]

    X_train = X[train_mask].copy()
    y_train = y[train_mask].copy()
    X_test = X[test_mask].copy()
    y_test = y[test_mask].copy()
    df_test = df_model.loc[test_mask].copy()

    train_median = X_train.median(numeric_only=True)
    X_train = X_train.fillna(train_median)
    X_test = X_test.fillna(train_median)

    event_model = StackingClassifier(
        estimators=[
            ("rf", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=RANDOM_STATE)),
            ("dt", DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
            ("lr", LogisticRegression(max_iter=2000, class_weight="balanced")),
        ],
        final_estimator=LogisticRegression(max_iter=2000, class_weight="balanced"),
        passthrough=True,
        cv=3
    )

    event_model.fit(X_train, y_train)
    y_prob = event_model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= decision_threshold).astype(int)

    df_test["predicted_flood"] = y_pred
    df_test["flood_probability"] = y_prob

    summary = []
    for area, group in df_test.groupby("area"):
        summary.append({
            "Area": area_labels.get(area, area),
            "Actual flood points": int(group["actual_flood"].sum()),
            "Predicted flood points": int(group["predicted_flood"].sum()),
            "Accuracy": accuracy_score(group["actual_flood"], group["predicted_flood"]),
            "Precision": precision_score(group["actual_flood"], group["predicted_flood"], zero_division=0),
            "Recall": recall_score(group["actual_flood"], group["predicted_flood"], zero_division=0),
            "F1": f1_score(group["actual_flood"], group["predicted_flood"], zero_division=0),
        })

    return pd.DataFrame(summary), df_test


## 11. Validate January 2023 and May 2023 Auckland flood events

This section evaluates the model on two real flood/storm windows: the January 2023 Auckland Anniversary Flood and the 9 May 2023 Auckland storm event. The results are combined into one summary table for easier comparison.


In [ ]:
event_windows = {
    "Auckland Anniversary Flood (January 2023)": {
        "start": pd.Timestamp("2023-01-27 00:00:00"),
        "end": pd.Timestamp("2023-01-29 23:59:59"),
        "decision_threshold": 0.50,
    },
    "Auckland Storm Event (9 May 2023)": {
        "start": pd.Timestamp("2023-05-08 00:00:00"),
        "end": pd.Timestamp("2023-05-10 23:59:59"),
        "decision_threshold": 0.50,
    },
}

all_event_summaries = []
event_predictions = {}

for event_name, event_info in event_windows.items():
    summary, predictions = validate_storm_event(
        cleaned_datasets,
        event_info["start"],
        event_info["end"],
        decision_threshold=event_info["decision_threshold"],
    )
    summary.insert(0, "Event", event_name)
    all_event_summaries.append(summary)
    event_predictions[event_name] = predictions

combined_event_summary = pd.concat(all_event_summaries, ignore_index=True)

# Backward-compatible variable names for later use
anniversary_predictions = event_predictions["Auckland Anniversary Flood (January 2023)"]
may_storm_predictions = event_predictions["Auckland Storm Event (9 May 2023)"]

combined_event_summary.round(3)


## 12. Plot validation results for both flood events

This section creates one clean plot per area for each event. Each plot shows river water level, the 99th percentile threshold, actual flood period, and predicted flood period.


In [ ]:
def plot_event_predictions(predictions_df, title_prefix="Event validation"):
    for area, group in predictions_df.groupby("area"):
        group = group.sort_values("time")
        threshold = group["flood_level_threshold"].iloc[0]

        fig, ax = plt.subplots(figsize=(13, 5))
        ax.plot(group["time"], group["river_level"], linewidth=1.8, label="River level")
        ax.axhline(threshold, linestyle="--", linewidth=1.5, label="99th percentile threshold")

        actual = group[group["actual_flood"] == 1]
        predicted = group[group["predicted_flood"] == 1]

        if not actual.empty:
            ax.axvspan(actual["time"].min(), actual["time"].max(), alpha=0.20, label="Actual flood period")

        if not predicted.empty:
            ax.axvspan(predicted["time"].min(), predicted["time"].max(), alpha=0.20, label="Predicted flood period")

        ax.set_title(f"{title_prefix}: {area_labels.get(area, area)}")
        ax.set_xlabel("Time")
        ax.set_ylabel("River water level (m)")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b %H:%M"))
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()


for event_name, predictions in event_predictions.items():
    plot_event_predictions(predictions, title_prefix=event_name)
